<a href="https://colab.research.google.com/github/jolineuichanco/DataAnalytics/blob/main/projects/project-2-prediction-challenge/Project_2_Scoring_Process.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project 2 Scoring Process**

This is a sample notebook to show you how you use the scoringData.csv to generate the predictions you need to submit.

## **Step 1:** Load and Prepare Training Data

In [41]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

url = 'https://raw.githubusercontent.com/jolineuichanco/DataAnalytics/main/projects/project-2-prediction-challenge/trainingData.csv'
data = pd.read_csv(url)

In [42]:
# Drop the "id" column, since it will not be used for modeling
data.drop(columns=["id"], inplace=True, errors="ignore")

# Encode categorical variables (GBM in sklearn requires numeric input)
label_encoders = {}
for col in data.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

## **Step 2:** Model Development

Your team will need to develop various models, then select the best one. So you need to modify Step 2 so that by the end, you have your own "final" model.

### Data pre-processing

E.g. Dropping columns that you don't want to use. Creating new features. Normalizing the data

In [43]:
# dropping columns not used in model
data.drop(columns=["toCoupon_GEQ5min"], inplace=True, errors="ignore")

# after data processing, you have your final X and y
predictors = [col for col in data.columns if col != "Y"]
X = data[predictors]
y = data["Y"]

### Training the model

Let's train a GBM model with default hyperparameters. You most likely will have a different *final* model.

In [46]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

from sklearn.model_selection import train_test_split

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Train GBM model
gbm_model = GradientBoostingClassifier(random_state=42)
gbm_model.fit(X_train, y_train)

# Evaluate on held-out test set
y_pred_test = gbm_model.predict(X_test)
print("=== Model Performance on Test Set ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_test):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_test):.4f}")
print()
print(classification_report(y_test, y_pred_test, target_names=["Rejected (0)", "Accepted (1)"]))


=== Model Performance on Test Set ===
Accuracy : 0.7190
F1 Score : 0.7672

              precision    recall  f1-score   support

Rejected (0)       0.70      0.60      0.65       858
Accepted (1)       0.73      0.81      0.77      1142

    accuracy                           0.72      2000
   macro avg       0.72      0.70      0.71      2000
weighted avg       0.72      0.72      0.72      2000



## **Step 3:** Use your Model to Score New Data

In [47]:
# load the scoring data
url = 'https://raw.githubusercontent.com/jolineuichanco/DataAnalytics/main/projects/project-2-prediction-challenge/scoringData.csv'
score_data = pd.read_csv(url)
score_data.drop(columns=["id"], inplace=True, errors="ignore")

# Apply the same label encoding to scoring data
for col in score_data.select_dtypes(include=["object"]).columns:
    if col in label_encoders:
        le = label_encoders[col]
        score_data[col] = score_data[col].astype(str).map(
            lambda x, le=le: le.transform([x])[0] if x in le.classes_ else -1
        )

=== Scoring Distribution ===
Rejected (0)     993
Accepted (1)    1691
Name: count, dtype: int64

Submission file saved with 2684 predictions.


### Applying pre-processing steps

You need to apply the same pre-processing that you did on your training to your scoring data

In [ ]:
# drop the same columns
score_data.drop(columns=["toCoupon_GEQ5min"], inplace=True, errors="ignore")

# if you did any normalization or feature engineering, you need to do those too on the scoring data

### Use your trained model to make predictions

In [49]:
# predict using the model you trained in Step 2
scoring = gbm_model.predict(score_data[predictors])

# as a check, I'm pring the scoring distribution
print("=== Scoring Distribution ===")
print(pd.Series(scoring).value_counts().sort_index().rename({0: "Rejected (0)", 1: "Accepted (1)"}))

=== Scoring Distribution ===
Rejected (0)     993
Accepted (1)    1691
Name: count, dtype: int64


### Save the predictions into a csv file

In [50]:
# download the provided template
url = 'https://raw.githubusercontent.com/jolineuichanco/DataAnalytics/main/projects/project-2-prediction-challenge/teamX_submission.csv'
submission = pd.read_csv(url)

# overwrite the "Y" column with your predictions
submission["Y"] = scoring

# write to csv
submission.to_csv("teamJU_submission.csv", index=False)
print(f"\nSubmission file saved with {len(submission)} predictions.")


Submission file saved with 2684 predictions.
